In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


class SelfAttentionBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.qkv_conv = nn.Linear(in_channels, out_channels * 3)
        self.proj = nn.Linear(out_channels, out_channels)
        self.scale = out_channels ** -0.5

    def forward(self, x):
        qkv = self.qkv_conv(x)  # [B, 3c]
        q, k, v = torch.chunk(qkv, 3, dim=-1)  # each: [B, c]
        attn = torch.bmm(q.unsqueeze(1), k.unsqueeze(2)) * self.scale  # [B, 1, 1]
        attn = torch.softmax(attn, dim=-1)
        out = torch.bmm(attn, v.unsqueeze(2)).squeeze(2)  # [B, c]
        out = self.proj(out)
        return out


def fuse_features(quantum, classical, method='concat'):
    """
    quantum: [N, Dq]
    classical: [N, Dc]
    """
    quantum = torch.tensor(quantum, dtype=torch.float32)
    classical = torch.tensor(classical, dtype=torch.float32)
    
    assert quantum.shape[0] == classical.shape[0], "Mismatch in sample count"
    fused = []

    if method == 'add':
        # Require same dimensionality
        min_dim = min(quantum.shape[1], classical.shape[1])
        q = quantum[:, :min_dim]
        c = classical[:, :min_dim]
        fused = q + c

    elif method == 'concat':
        fused = torch.cat([quantum, classical], dim=1)

    elif method == 'attention':
        inter = torch.cat([quantum, classical], dim=1)
        gate = nn.Sequential(
            nn.Linear(inter.shape[1], inter.shape[1] // 2),
            nn.SiLU(),
            nn.Linear(inter.shape[1] // 2, quantum.shape[1]),
            nn.Sigmoid()
        )
        attn = gate(inter)
        fused = quantum * attn + classical * (1 - attn)

    elif method == 'self_attention':
        sa_block = SelfAttentionBlock(in_channels=quantum.shape[1] + classical.shape[1],
                                      out_channels=quantum.shape[1])
        sa_block.eval()  # Disable dropout or batchnorm if any
        with torch.no_grad():
            inter = torch.cat([quantum, classical], dim=1)
            fused = sa_block(inter)

    elif method == 'dot_product':
        q_norm = F.normalize(quantum, p=2, dim=1)
        c_norm = F.normalize(classical, p=2, dim=1)
        sim = torch.sum(q_norm * c_norm, dim=1, keepdim=True)  # [N, 1]
        weight = (sim + 1) / 2  # scale from [-1,1] to [0,1]
        fused = quantum * weight + classical * (1 - weight)

    return fused.numpy()


# ===== Main procedure =====
if __name__ == '__main__':
    import os
    import pandas as pd

    fusion_type = 'self_attention'  # change here

    quantum_path = 'HybridEmbeddingsForBreastCancerDetection/embeddings/quantum'
    classical_path = 'HybridEmbeddingsForBreastCancerDetection/embeddings/classical'
    save_path = f'HybridEmbeddingsForBreastCancerDetection/embeddings/fused/{fusion_type}'
    os.makedirs(save_path, exist_ok=True)

    quantum_feat = np.load(os.path.join(quantum_path, 'features.npy'))  # (546, 196)
    classical_feat = np.load(os.path.join(classical_path, 'features.npy'))  # (546, 512)
    labels = np.load(os.path.join(classical_path, 'labels.npy'))  # (546, 1)

    fused = fuse_features(quantum_feat, classical_feat, method=fusion_type)

    np.save(os.path.join(save_path, 'features.npy'), fused)
    np.save(os.path.join(save_path, 'labels.npy'), labels)

    # Save to CSV
    df = pd.DataFrame(fused)
    df['label'] = labels.flatten()
    df.to_csv(os.path.join(save_path, 'features.csv'), index=False)

    print(f">>> Fusion done with method '{fusion_type}'. Saved to {save_path}")
